In [ ]:
from datetime import datetime, timezone
import pathlib
from zoneinfo import ZoneInfo

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import requests

pio.templates.default = "plotly_dark"

import aio_stats

In [ ]:
data_file = pathlib.Path("~/Documents/Autolux-20250223-2114.csv")
tzinfo = ZoneInfo("America/Phoenix")

In [ ]:
aio_file = aio_stats.AioFile(data_file)
data_records = aio_file.read_data()
data = aio_file.transform_data(data_records, tzinfo.key)

In [ ]:
stats = aio_stats.StatsMaker()
stats.create_dataframe(data, "lux")

In [ ]:
start = datetime.fromisoformat("2025-02-18T00:00:00").astimezone(tzinfo)
end = datetime.fromisoformat("2025-02-19T00:00:00").astimezone(tzinfo)
start_utc = datetime.fromtimestamp(start.timestamp(), timezone.utc)

In [ ]:
stats.filter_time(start, end)

In [ ]:
info_file = pathlib.Path("~/Documents/location_info.json").expanduser()
with info_file.open() as ifile:
    info = json.load(ifile)
latitude = info["latitude"]
longitude = info["longitude"]
helios_ws = info["helios_ws"]
payload = {"lat": latitiude, "lon": longitude, "cdatetime": start_utc.timestamp(), "tz": tzinfo.key}
response = requests.get(helios_ws, params=payload)

In [ ]:
print(response.status_code)
sky_transitions = response.json()

In [ ]:
for key in sky_transitions:
    timestamp = sky_transitions[key]
    sky_transitions[key] = datetime.fromtimestamp(timestamp).astimezone(tzinfo)

In [ ]:
c = aio_stats.AioClient()
nd = c.fetch_data("living-room.notifier", 50)
notifier = c.transform_data(nd, tzinfo.key)

In [ ]:
n1 = stats.df.loc[start:sky_transitions["astronomical_dawn"]]
n2 = stats.df.loc[lamp_time:end]
nighttime = pd.concat([n1, n2])

In [ ]:
lamp_time = None
from datetime import timedelta
for value in notifier:
    if start.date() == value[0].date():
        lamp_time = value[0]
if lamp_time is None:
    lamp_time = sky_transitions["sunset"]
else:
    lamp_time += timedelta(seconds=5*60) 

In [ ]:
daytime = stats.df.loc[sky_transitions["sunrise"]:lamp_time]

In [ ]:
layout = dict(height=525, width=700)
fig = go.Figure(layout=layout)
binning = dict(start=0, end=daytime.max()+10, size=1)
d_trace = go.Histogram(x=daytime.lux, xbins=binning, name="day")
n_trace = go.Histogram(x=nighttime.lux, xbins=binning, name="night")

In [ ]:
fig.add_trace(d_trace)
fig.add_trace(n_trace)

In [ ]:
daytime.max()

In [ ]:
daytime.mean()

In [ ]:
daytime.min()

In [ ]:
daytime.median()

In [ ]:
f = pathlib.Path("test.parquet")
stats.df.to_parquet(f, engine="pyarrow")